# Ноутбук для проведения вычислительных экспериментов по оценке точности алгоритма ADD (жадное добавление)

In [ ]:
import os

import osmnx as ox
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import tabulate
import contextily as ctx

# Версии библиотек
print('OSMnx:', ox.__version__,
      '\nGeoPandas:', gpd.__version__,
      '\nPandas:', pd.__version__,
      '\nContextily', ctx.__version__,
      )

ox.settings.default_access = ''

OSMnx: 2.0.1 
GeoPandas: 1.0.1 
Pandas: 2.2.2 
Contextily 1.6.2


In [2]:
DATA_PATH = '_data/add_check_data/'

# Основной цикл

In [84]:
settlements = pd.read_csv(os.path.join(DATA_PATH, 'settlements.csv'), index_col=0)
settlements['joined'] = settlements['joined'].str.strip()
cities_list = settlements['joined'].unique().tolist()

print('Количество городов:', len(cities_list))
cities_list

Количество городов: 93


['Обнинск, Калужская область',
 'город Евпатория, Республика Крым',
 'Сызрань, Самарская область',
 'Кызыл, Республика Тыва',
 'Южно-Сахалинск, Сахалинская область',
 'Старый Оскол, Белгородская область',
 'Невинномысск, Ставропольский край',
 'Сергиев Посад, Московская область',
 'Дербент, Республика Дагестан',
 'Новочеркасск, Ростовская область',
 'Ковров, Владимирская область',
 'Нижнекамск, Республика Татарстан',
 'Бердск, Новосибирская область',
 'Королев, Московская область',
 'Майкоп, Республика Адыгея',
 'Балаково, Саратовская область',
 'Волгодонск, Ростовская область',
 'Благовещенск, Амурская область',
 'Черкесск, Карачаево-Черкесская республика',
 'Шахты, Ростовская область',
 'Жуковский, Московская область',
 'Ногинск, Московская область',
 'Железногорск, Курская область',
 'Елец, Липецкая область',
 'Великий Новгород, Новгородская область',
 'Батайск, Ростовская область',
 'Балашиха, Московская область',
 'Альметьевск, Республика Татарстан',
 'Ангарск, Иркутская область',

In [ ]:
# cities_list = [
#     'Ачинск',
#     'Канск',
#     'Железногорск',
#     'Братск',
#     'Бердск',
#     'Novokuznetsk'
# ]

In [ ]:
from fire_units.metrics import CoverIndexBuilding
from fire_units.settings import BUILDING_FIELDS, DEFAULT_SPEEDS
from genesis.lscp import LSCP_ADD, drop_trash_points
from genesis.states import FirstArrivalUnitState, get_atm
from graphs.speeds import kmh_to_mm, set_graph_travel_times


def get_bounds(city_name, city_name_full, layer_name='bounds'):
    '''
    Получение полигона границ
    '''
    path = f'{DATA_PATH}{city_name}.gpkg'
    try:
        gdf = gpd.read_file(path, layer=layer_name)
    except:
        gdf = ox.geocode_to_gdf(city_name_full)
        if gdf is None:
            raise ValueError(f'Данные для города {city_name_full} не получены!')
        gdf.to_file(path, layer=layer_name)

    print(f'Границы города {city_name} получены!')
    return gdf
        
def get_g(city_name, city_name_full, simplify = False, retain_all = False):
    '''
    Получение графа улично-дорожной сети
    '''
    path = f'{DATA_PATH}{city_name}.ml'
    if os.path.exists(path):
        G = ox.load_graphml(path)
    else:
        G = ox.graph_from_place(city_name_full,
                                network_type = 'drive_service',
                                simplify     = simplify,
                                retain_all   = retain_all,
                                )
        if G is None:
            raise ValueError(f'ГДС города {city_name} не получен!')
        
        set_graph_travel_times(
            G,
            speeds         = DEFAULT_SPEEDS,
            morph_function = kmh_to_mm)
        
        ox.save_graphml(G, path)

    print(f'ГДС города {city_name} получен! Времена следования установлены!')
    return G

def get_buildings(city_name, city_bounds, layer_name='buildings'):
    '''
    Загрузка данных о зданиях
    '''
    path = f'{DATA_PATH}{city_name}.gpkg'
    try:
        buildings = gpd.read_file(path, layer=layer_name)
    except:
        buildings = ox.features_from_polygon(city_bounds, tags={'building': True})
        if buildings is None:
            raise ValueError(f'Здания города {city_name} не получены!')
        # Оставляем только данные с геометрией Полигон или Мультиполигон
        buildings = buildings[buildings.geometry.apply(lambda x: x.geom_type).isin(['Polygon', 'MultiPolygon'])]
        # Оставляем только заданные поля (и при этом имеющиеся в списке полученных)
        fields = list(set(buildings.columns) & set(BUILDING_FIELDS))
        buildings = buildings[fields]
        # Удаляем индекс по типу элемента
        buildings = buildings.reset_index().drop('element', axis=1)
        # Устанавливаем индекс 'id'
        buildings = buildings.set_index('id')

        buildings.to_file(path, layer=layer_name)

    print(f'Здания города {city_name} получены! Всего {len(buildings)} зданий.')

    return buildings

def add_buildings_node(G, buildings, drop_node_duplicates = True):
    '''
    Установка ближайшего узла для зданий
    '''
    centroids = ox.projection.project_gdf(buildings).geometry.centroid
    buildings['node'] = ox.distance.nearest_nodes(ox.project_graph(G), 
                                              centroids.x,
                                              centroids.y
                                              )
    # Отбрасываем дублирующиеся здания (по узлам)
    if drop_node_duplicates:
        t =len(buildings)
        buildings = buildings.drop_duplicates(subset='node')  #.set_index('node') #.sort_index()
        print(f'Размер набора зданий без дубликатов в узлах {len(buildings)} (из {t})')

    print('Здания с ближайшими узлами ГДС сопоставлены.')
    return buildings
    
def plot_map(borders, g_nodes, best_nodes, best_nodes2, city_name):
    ## Улучшенная печать карты
    """
    Печать карты с границами города, узлами "можно удалить" и "достаточное размещение"
    """
    # Настройка стиля графика
    plt.style.use('bmh')

    fig, ax = plt.subplots(figsize=(10, 10))
    borders.plot(ax=ax, color='#fcfcd9', edgecolor='black', linewidth=2, alpha=0.7, zorder=1)

    # Отображение узлов
    g_nodes.loc[best_nodes.keys()].plot(ax=ax, facecolors='r', edgecolors='k', markersize=50, label='Можно удалить')
    g_nodes.loc[best_nodes2.keys()].plot(ax=ax, facecolors='g', edgecolors='k', markersize=50, label='Достаточное размещение')


    # Подписи к узлам
    offset = (float(borders.bounds.iloc[0]['maxy']) - float(borders.bounds.iloc[0]['miny'])) / 100
    for node, unit in best_nodes.items():
        g_node_geometry = g_nodes.loc[node].geometry
        ax.text(
            g_node_geometry.x,
            g_node_geometry.y,
            str(unit),
            fontsize=10, 
            color='black',
            ha='center',
            va='bottom',
            position = (g_node_geometry.x, g_node_geometry.y + offset),
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='k', boxstyle='round,pad=0.2'),
            zorder=5
        )

    # Добавить подложку OSM
    ctx.add_basemap(ax, 
                    crs=g_nodes.crs, 
                    source=ctx.providers.OpenStreetMap.Mapnik, 
                    alpha=0.7, 
                    zorder=0)

    ax.set_axis_off()
    ax.legend(loc='upper left', fontsize=12, frameon=True)
    ax.set_title(city_name, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{DATA_PATH}imgs/{city_name}.png')
    plt.show()


def save_results(city_name, best_nodes, best_metric, best_nodes2, best_metric2):
    path = f'{DATA_PATH}results.csv'
    try:
        results = pd.read_csv(path, index_col='Город')
        results.loc[city_name] = {
            'ПСЧ до': len(best_nodes),
            'ИП-10 до': best_metric,
            'ПСЧ после': len(best_nodes2),
            'ИП-10 после': best_metric2
        }
        # print(tabulate.tabulate(results))
    except:
        results = pd.DataFrame(
            {
                'Город': [city_name],
                'ПСЧ до': len(best_nodes),
                'ИП-10 до': best_metric,
                'ПСЧ после': len(best_nodes2),
                'ИП-10 после': best_metric2                
            }
        )
        results.set_index('Город', inplace=True)

    results.to_csv(path, index=True)



def check_step(city_name_full, aim_ip=95, drop_node_duplicates=True, buildings_size=10000):

    # Уточняем имена населенных пунктов
    city_name      = city_name_full.split(', ')[0]
    city_name_full = city_name_full + ', Россия'

    print(f'Расчет для города "{city_name}"')

    # 1. Загрузка данных
    # Границы города
    borders = get_bounds(city_name, city_name_full)
    city_bounds = borders.union_all()

    # Загрузка ГДС
    G = get_g(city_name, city_name_full)

    # Загрузка зданий
    buildings = get_buildings(city_name, city_bounds)
    buildings = add_buildings_node(G, buildings, drop_node_duplicates = drop_node_duplicates)


    # 2. Расчет ADD
    # Матрица
    if len(buildings) > buildings_size:
        print(f'Объем выборки: {buildings_size} зданий')
        buildings_set = buildings.sample(buildings_size)
    else:
        print(f'Объем выборки: {len(buildings)} зданий (все)')
        buildings_set = buildings
    matrix = get_atm(G,
                 data             = buildings_set,
                 cutoff           = 10,
                 )

    # 3. Сборка и инициализация алгоритма
    ## Выбор алгоритма моделирования параметров прибытия
    state = FirstArrivalUnitState()

    ## Выбор метрики
    metric = CoverIndexBuilding(buildings_set, ip_val=10) # Используется метрика ИП-10. Для ADD метрика выполняет роль оценки

    ## Функция остановки
    stop_case = lambda value, **kwargs: value >= aim_ip

    ## Функция выполняемая при добавлении новой ПСЧ
    after_mclp = lambda best_metric, dynamic_nodes, **kwargs: print('Подразделений:', len(dynamic_nodes), ' ИП-10:', best_metric)

    ## Сборка и инициализация алгоритма ADD
    add = LSCP_ADD(  
        state_function      = state,
        matrix              = matrix,
        metric_function     = metric,
        stop_case_function  = stop_case,
        after_mclp_function = after_mclp,
    )


    # 3. Расчет
    best_nodes, best_metric = add(
            env          = G,
        )


    # 4. Печать результатов
    print('\t', 'ИТОГ:')
    print('\t', 'Подразделений:', len(best_nodes), ' ИП-10:', best_metric)

    ## Получение набора данных узлов графа, для сопоставления с полученным лучшим узлом
    g_nodes = ox.graph_to_gdfs(G, edges=False)
    g_nodes.to_file(f'{DATA_PATH}{city_name}.gpkg', layer='add')


    # 5. Отброс мусорных размещений
    best_nodes2, best_metric2 = drop_trash_points(G,
                    state_function      = state,
                    metric_function     = metric,
                    stop_case_function  = stop_case,
                    dynamic_nodes       = best_nodes,
                      )
    print('\t', 'После отброса:')
    print('\t', 'Подразделений:', len(best_nodes2), ' ИП-10:', best_metric2)

    # Печать итога
    df = pd.DataFrame({'Подразделений': [len(best_nodes), len(best_nodes2)],
                       'ИП-10': [best_metric, best_metric2]}, index = ['ADD', 'После отброса'])
    print(tabulate.tabulate(df, headers='keys', tablefmt='grid', showindex=True))

    # 6. Сохранение результатов
    save_results(city_name, best_nodes, best_metric, best_nodes2, best_metric2)

    ## Печать карты
    plot_map(borders, g_nodes, best_nodes, best_nodes2, city_name_full)



# НЕПОСРЕДСТВЕННО ВЫЧИСЛЕНИЯ
# check_step(cities_list[0], aim_ip=100, drop_node_duplicates=True, buildings_size=100)

for i, city_name in enumerate(cities_list):
    print('\t', f'{i} из {len(cities_list)}')

    check_step(city_name, aim_ip=100, buildings_size=500, drop_node_duplicates=True)

	 0 из 93
Расчет для города "Обнинск"
Границы города Обнинск получены!
ГДС города Обнинск получен! Времена следования установлены!
Здания города Обнинск получены! Всего 5330 зданий.
Размер набора зданий без дубликатов в узлах 2778 (из 5330)
Здания с ближайшими узлами ГДС сопоставлены.
Объем выборки: 2778 зданий (все)


KeyboardInterrupt: 

In [ ]:
plt.plot([1, 2, 3], [4, 5, 1])
plt.title("График")
plt.savefig(f'{DATA_PATH}imgs/{city_name}.png')
plt.show()